In [ ]:
"""
The purpose of this Jupyter notebook is to perform the
training-validation-test split for positive-unlabeled (PU) learning.
"""

'\nThe purpose of this Jupyter notebook is to perform the\ntraining-validation-test split for positive-unlabeled (PU) learning.\n\n\nRemoving genes involved in the PPI training set so as to prevent label\nleakage/target leakage is a bit overcautious for the following reason:\nWhile it is true that the PPI prediction model has some prior knowledge\nabout VACV biology, it primarily encodes the interaction propensity with\nVACV proteins. This is not the same as the host factor status, as the\nhost factor status depends on other factors as well.\n'

In [ ]:
import os
import ast
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import jensenshannon

# Loading Data

In [ ]:
# Load the screen TSV file into a Pandas DataFrame
screen_path = (
    "data_for_training_validation_test_split/Dharmacon_pooled_G1_G2_"
    "screening_plates_subset_control-based_Z-scored.tsv"
)

screen_df = pd.read_csv(
    screen_path,
    sep="\t"
)

In [ ]:
# Load the TSV file mapping confirmed/reliable VACV host factors to
# their categories
all_hf_with_category_path = (
    "data_for_training_validation_test_split/all_host_factors_with_"
    "categories.tsv"
)

all_hf_with_category_df = pd.read_csv(
    all_hf_with_category_path,
    sep="\t"
)

# Extracting Features from the Screen DataFrame

In [6]:
# Filter the screen DataFrame to retain only rows associated with
# proteins
# Also exclude controls by selecting the well type `POOLED_SIRNA`
screen_df = screen_df.loc[
    (screen_df["WellType"] == "POOLED_SIRNA")
    &
    (screen_df["UniProt_IDs"].notna())
]

assert screen_df["Name"].notna().all(), (
    "There indeed are empty cells in the `Name` column!"
)

In [7]:
# Extract the columns of interest from the screen DataFrame
# These are the gene name (`Name`), the UniProt accessions
# (`UniProt_IDs`), the early intensity
# (`dIntensity_cPathogen_eMean_oVoronoiCells_nZScore`), the late
# intensity (`dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore`) and
# the cell count (`eCount_oCells_nZScore`)
screen_df = screen_df[[
    "Name",
    "UniProt_IDs",
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
    "eCount_oCells_nZScore"
]]

In [8]:
# The screen TSV file contains replicates for each gene
# Thus, for each interrogated gene, the mean feature is computed
# To this end, the DataFrame is grouped by the gene name (and also by
# the UniProt accessions)
mean_features_df = (
    screen_df
    .groupby(["Name", "UniProt_IDs"], as_index=False)
    .mean()
)

In [9]:
# Also explode the `UniProt_IDs` column
mean_features_df["UniProt_IDs"] = (
    mean_features_df["UniProt_IDs"].str.split(";")
)

mean_features_df = mean_features_df.explode("UniProt_IDs")

In [ ]:
# Save the DataFrame with the mean features to disk
mean_features_df.to_csv(
    os.path.join(
        "data_for_training_validation_test_split",
        "screen_subset_mean_features.tsv"
    ),
    sep="\t",
    index=False
)

# Defining a Procedure for the Training-Validation-Test Split

In [ ]:
# Define the procedure for the training-validation-test split

# -----------------------------
# 0. Phenotype clustering
# -----------------------------
def compute_pheno_clusters(df, n_clusters=10, seed=42):
    X = np.vstack(df["phenotype_vec"].values)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)

    df = df.copy()
    df["pheno_cluster"] = clusters
    return df


# ----------------------------------------
# 1. Collapsing the rows to the gene level
# ----------------------------------------
def collapse_to_gene_level(df):
    rows = []

    for gene, subdf in df.groupby("gene"):
        row = {}

        row["gene"] = gene
        row["label"] = int(subdf["label"].max())

        # HF class
        hf_classes = subdf["hf_class"].dropna().unique()
        if row["label"] == 1:
            row["hf_class"] = hf_classes[0] if len(hf_classes) == 1 else "mixed"
        else:
            row["hf_class"] = "unlabeled"

        # Proteins
        row["protein_ids"] = subdf["protein_id"].tolist()

        # Phenotype vector (unchanged)
        row["phenotype_vec"] = subdf["phenotype_vec"].iloc[0]

        # Propagate phenotype cluster
        row["pheno_cluster"] = subdf["pheno_cluster"].mode()[0]

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# 2. Build connected components
# -----------------------------
def build_gene_protein_components(gene_df):
    gene_to_proteins = {}
    protein_to_genes = defaultdict(set)

    for _, row in gene_df.iterrows():
        g = row["gene"]
        prots = row["protein_ids"]

        gene_to_proteins[g] = set(prots)
        for p in prots:
            protein_to_genes[p].add(g)

    visited = set()
    components = []

    def bfs(start_gene):
        stack = [start_gene]
        genes = set()
        proteins = set()

        while stack:
            node = stack.pop()

            if node in visited:
                continue
            visited.add(node)

            if node in gene_to_proteins:
                genes.add(node)
                for p in gene_to_proteins[node]:
                    if p not in visited:
                        stack.append(p)
            else:
                proteins.add(node)
                for g in protein_to_genes[node]:
                    if g not in visited:
                        stack.append(g)

        return genes, proteins

    for gene in gene_to_proteins:
        if gene not in visited:
            genes, proteins = bfs(gene)
            components.append({
                "genes": list(genes),
                "proteins": list(proteins)
            })

    return components


# -----------------------------
# 3. Build component dataframe
# -----------------------------
def build_component_df(components, gene_df):
    rows = []

    for i, comp in enumerate(components):
        subdf = gene_df[gene_df["gene"].isin(comp["genes"])]

        label = int(subdf["label"].max())

        if label == 1:
            hf_classes = subdf["hf_class"].unique()
            hf_class = hf_classes[0] if len(hf_classes) == 1 else "mixed"
        else:
            hf_class = "unlabeled"

        pheno_cluster = subdf["pheno_cluster"].mode()[0]

        rows.append({
            "component_id": i,
            "genes": comp["genes"],
            "label": label,
            "hf_class": hf_class,
            "pheno_cluster": pheno_cluster,
            "size": len(comp["genes"])
        })

    return pd.DataFrame(rows)


# -----------------------------
# 4. Split
# -----------------------------
def split_components_balanced(comp_df, seed=42):
    np.random.seed(seed)

    comp_df = comp_df.sample(frac=1, random_state=seed)
    comp_df = comp_df.sort_values("size", ascending=False).reset_index(drop=True)

    splits = {
        "train": {"rows": [], "size": 0},
        "val": {"rows": [], "size": 0},
        "test": {"rows": [], "size": 0},
    }

    target_ratios = {"train": 0.7, "val": 0.1, "test": 0.2}
    total_size = comp_df["size"].sum()
    target_sizes = {k: v * total_size for k, v in target_ratios.items()}

    # Seed splits
    split_names = list(splits.keys())
    for i, (_, comp) in enumerate(comp_df.iloc[:3].iterrows()):
        splits[split_names[i]]["rows"].append(comp)
        splits[split_names[i]]["size"] += comp["size"]

    remaining = comp_df.iloc[3:]

    for _, comp in remaining.iterrows():
        deficits = {
            s: target_sizes[s] - splits[s]["size"]
            for s in splits
        }
        best_split = max(deficits, key=deficits.get)

        splits[best_split]["rows"].append(comp)
        splits[best_split]["size"] += comp["size"]

    return (
        pd.DataFrame(splits["train"]["rows"]),
        pd.DataFrame(splits["val"]["rows"]),
        pd.DataFrame(splits["test"]["rows"]),
    )


# -----------------------------
# 5. Expand to gene-level
# -----------------------------
def expand_components(comp_split, gene_df):
    genes = []
    for _, row in comp_split.iterrows():
        genes.extend(row["genes"])
    return gene_df[gene_df["gene"].isin(genes)].copy()


# -----------------------------
# 6. Enforce P/U ratio
# -----------------------------
def enforce_pu_ratio(df, target_ratio=5, seed=42):
    np.random.seed(seed)

    pos = df[df["label"] == 1]
    unl = df[df["label"] == 0]

    n_pos = len(pos)
    n_unl_target = int(n_pos * target_ratio)

    if n_unl_target >= len(unl):
        return df

    sampled_unl = []

    for cluster, subdf in unl.groupby("pheno_cluster"):
        frac = len(subdf) / len(unl)
        n_cluster = int(frac * n_unl_target)

        sampled_unl.append(
            subdf.sample(n=min(n_cluster, len(subdf)), random_state=seed)
        )

    sampled_unl = pd.concat(sampled_unl)

    return pd.concat([pos, sampled_unl])


# -----------------------------
# 7. Logging
# -----------------------------
def log_class_balance(df, name):
    print(f"\n[{name}]")
    print("Total:", len(df))
    print("Positives:", df["label"].sum())
    print("Unlabeled:", (df["label"] == 0).sum())
    print("HF class distribution:")
    print(df["hf_class"].value_counts(dropna=False))


def phenotype_distribution(df):
    return df["pheno_cluster"].value_counts(normalize=True).sort_index()


def compare_distributions(train_df, val_df, test_df):
    print("\n[Phenotype Distribution Similarity]")

    train_dist = phenotype_distribution(train_df)
    val_dist = phenotype_distribution(val_df)
    test_dist = phenotype_distribution(test_df)

    # Align indices
    all_idx = sorted(set(train_dist.index) | set(val_dist.index) | set(test_dist.index))
    train_dist = train_dist.reindex(all_idx, fill_value=0)
    val_dist = val_dist.reindex(all_idx, fill_value=0)
    test_dist = test_dist.reindex(all_idx, fill_value=0)

    print("JS(train || val): ", jensenshannon(train_dist, val_dist))
    print("JS(train || test):", jensenshannon(train_dist, test_dist))


# -----------------------------
# 9. Full pipeline
# -----------------------------
def create_splits(
    df,
    n_pheno_clusters=10,
    pu_ratio=5,
    seed=42
):
    print("Step 1: Phenotype clustering...")
    df = compute_pheno_clusters(df, n_pheno_clusters, seed)

    print("Step 2: Collapse to gene level...")
    gene_df = collapse_to_gene_level(df)

    print("Step 3: Building components...")
    components = build_gene_protein_components(gene_df)
    comp_df = build_component_df(components, gene_df)
    print(comp_df["size"].describe())

    print("Step 4: Balanced component splitting...")
    train_c, val_c, test_c = split_components_balanced(comp_df, seed)

    train_df = expand_components(train_c, gene_df)
    val_df = expand_components(val_c, gene_df)
    test_df = expand_components(test_c, gene_df)

    print("Step 5: Enforcing P/U ratio...")
    train_df = enforce_pu_ratio(train_df, pu_ratio, seed)
    val_df = enforce_pu_ratio(val_df, pu_ratio, seed)
    test_df = enforce_pu_ratio(test_df, pu_ratio, seed)

    assert train_df["gene"].nunique() == len(train_df)
    assert val_df["gene"].nunique() == len(val_df)
    assert test_df["gene"].nunique() == len(test_df)

    assert set(train_df["gene"]).isdisjoint(val_df["gene"])
    assert set(train_df["gene"]).isdisjoint(test_df["gene"])
    assert set(val_df["gene"]).isdisjoint(test_df["gene"])

    # Logging
    log_class_balance(train_df, "TRAIN")
    log_class_balance(val_df, "VAL")
    log_class_balance(test_df, "TEST")

    compare_distributions(train_df, val_df, test_df)

    return train_df, val_df, test_df

# Constructing the DataFrame for the Train-Validation-Test Split

In [ ]:
mean_features_df = pd.read_csv(
    os.path.join(
        "data_for_training_validation_test_split",
        "screen_subset_mean_features.tsv"
    ),
    sep="\t"
)

In [ ]:
df_for_split = mean_features_df.copy()

df_for_split["phenotype_vec"] = df_for_split.apply(
    lambda row: [
        row["dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"],
        row["dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"],
        row["eCount_oCells_nZScore"]
    ],
    axis=1
)

df_for_split.drop(
    [
        "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
        "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
        "eCount_oCells_nZScore"
    ],
    axis=1,
    inplace=True
)

In [ ]:
# Rename the first two columns
df_for_split.rename(
    columns={
        "Name": "gene",
        "UniProt_IDs": "protein_id"
    },
    inplace=True
)

In [ ]:
# Add a `label` column to the DataFrame
# The `label` column indicates whether the respective gene/protein is a
# known VACV host factor or not
# Accordingly, it can assume two values (1 for known host factors, i.e.
# positive, and 0 for unlabeled instances)
all_vacv_host_factors = all_hf_with_category_df["host_factor"].unique()

df_for_split["label"] = (
    df_for_split["gene"].isin(all_vacv_host_factors).astype(int)
)

In [ ]:
# Lastly, introduce the `hf_class` column
# The `hf_class` column, as its name already implies, indicates for
# known VACV host factors, i.e. rows with `label == 1`, the host factor
# class
# This column can assume three values, which are "proviral", "antiviral"
# and None
df_for_split_all_hf = pd.merge(
    df_for_split,
    all_hf_with_category_df,
    left_on="gene",
    right_on="host_factor",
    how="left"
)

# Drop the `host_factor` column
df_for_split_all_hf.drop(
    "host_factor",
    axis=1,
    inplace=True
)

In [15]:
# Rename the `category` column into `hf_class`
df_for_split_all_hf.rename(
    columns={"category": "hf_class"},
    inplace=True
)

In [ ]:
# Save the DataFrame for the split to disk
df_for_split_all_hf.to_csv(
    os.path.join(
        "data_for_training_validation_test_split",
        "DataFrame_for_train_validation_test_split.tsv"
    ),
    sep="\t",
    index=False
)

# Training-Validation-Test Split for All Host Factors

In [ ]:
# Load the DataFrame to subject to the train-validation-test split
df_for_split_all_hf = pd.read_csv(
    os.path.join(
        "data_for_training_validation_test_split",
        "DataFrame_for_train_validation_test_split.tsv"
    ),
    sep="\t",
    converters={"phenotype_vec": ast.literal_eval}
)

In [ ]:
train_df, val_df, test_df = create_splits(
    df_for_split_all_hf,
    pu_ratio=5
)

train_df.to_csv(
    os.path.join(
        "dataset_prior_to_formatting",
        "PU_training_set.tsv"
    ),
    sep="\t",
    index=False
)

val_df.to_csv(
    os.path.join(
        "dataset_prior_to_formatting",
        "PU_validation_set.tsv"
    ),
    sep="\t",
    index=False
)

test_df.to_csv(
    os.path.join(
        "dataset_prior_to_formatting",
        "PU_test_set.tsv"
    ),
    sep="\t",
    index=False
)

Step 1: Phenotype clustering...
Step 2: Collapse to gene level...
Step 3: Building components...
count    17367.000000
mean         1.017562
std          0.225574
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         12.000000
Name: size, dtype: float64
Step 4: Balanced component splitting...
Step 5: Enforcing P/U ratio...

[TRAIN]
Total: 1046
Positives: 175
Unlabeled: 871
HF class distribution:
hf_class
unlabeled    871
mixed        175
Name: count, dtype: int64

[VAL]
Total: 104
Positives: 18
Unlabeled: 86
HF class distribution:
hf_class
unlabeled    86
mixed        18
Name: count, dtype: int64

[TEST]
Total: 294
Positives: 50
Unlabeled: 244
HF class distribution:
hf_class
unlabeled    244
mixed         50
Name: count, dtype: int64

[Phenotype Distribution Similarity]
JS(train || val):  0.08055088034157483
JS(train || test): 0.033303599883028515
